# Notebook 11.5 — Constrained Adiabatic CZ

This notebook upgrades the adiabatic / dressed schedule by adding **physics-aware constraints**.

Instead of sweeping with generic ramps, we now:

- use smoother endpoint schedules,
- tie the detuning scale to the interaction strength $V$,
- score candidates using a **constraint-aware objective** that combines:
  - phase-corrected fidelity,
  - leakage penalty,
  - entangling-phase error.

This is the bridge between:
- Notebook 11: *explore adiabatic schedules*
- Notebook 12: *optimize them automatically*


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, qeye, tensor, sigmax, sesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Constraint-aware schedules

In [ ]:
def omega_schedule(t, T, omega_max):
    # smoother, flatter edges than sin^2
    s = t / T
    return omega_max * (s * (1.0 - s)) ** 2 * 16.0  # normalized to peak near omega_max

def delta_schedule(t, T, V, alpha=1.0, delta0=0.0):
    # tie detuning scale to interaction strength
    s = t / T
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * s))


## Time-dependent Hamiltonian

In [ ]:
H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0=0.0):
    def coeff_omega(t, args=None):
        return omega_schedule(t, T=T, omega_max=omega_max)

    def coeff_delta(t, args=None):
        return delta_schedule(t, T=T, V=V, alpha=alpha, delta0=delta0)

    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Effective-unitary helpers

In [ ]:
def run_adiabatic_evolution(psi0, T, omega_max, alpha, V, delta0=0.0, n_steps=320):
    H = build_time_dependent_hamiltonian(T=T, omega_max=omega_max, alpha=alpha, V=V, delta0=delta0)
    times = np.linspace(0.0, T, n_steps)
    result = sesolve(H, psi0, times)
    return result.states[-1]

def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)

def build_effective_unitary(T, omega_max, alpha, V, delta0=0.0):
    columns = []
    for psi0 in basis_states:
        psi_final = run_adiabatic_evolution(
            psi0,
            T=T,
            omega_max=omega_max,
            alpha=alpha,
            V=V,
            delta0=delta0,
        )
        columns.append(state_to_column(psi_final))
    return np.column_stack(columns)


## Gate metrics and constraint score

In [ ]:
def process_fidelity(U_eff, U_target=U_cz):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def phase_corrected_target(U_eff):
    phi_ent = entangling_phase(U_eff)
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)

def phase_corrected_fidelity(U_eff):
    return process_fidelity(U_eff, U_target=phase_corrected_target(U_eff))

def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))

def phase_error(U_eff):
    return abs(abs(entangling_phase(U_eff)) - np.pi)

def composite_score(U_eff, w_phase=0.8, w_leak=0.45):
    pc = phase_corrected_fidelity(U_eff)
    leak = leakage_metric(U_eff)
    perr = phase_error(U_eff)
    return float(pc - w_leak * leak - w_phase * perr)


## Compare a few constrained candidates

In [ ]:
V = 4.0
cases = [
    ('short', 12.0, 1.0, 0.5),
    ('medium', 24.0, 1.0, 0.5),
    ('slower stronger', 32.0, 1.2, 0.6),
]

for label, T, omega_max, alpha in cases:
    U_eff = build_effective_unitary(T=T, omega_max=omega_max, alpha=alpha, V=V, delta0=0.0)
    print(label)
    print(f'  T={T:.3f}, omega_max={omega_max:.3f}, alpha={alpha:.3f}')
    print(f'  raw fidelity:             {process_fidelity(U_eff):.6f}')
    print(f'  phase-corrected fidelity: {phase_corrected_fidelity(U_eff):.6f}')
    print(f'  entangling phase / pi:    {entangling_phase(U_eff)/np.pi:.6f}')
    print(f'  leakage:                  {leakage_metric(U_eff):.6f}')
    print(f'  composite score:          {composite_score(U_eff):.6f}')
    print()


## Effective unitary for a constrained candidate

In [ ]:
T_demo = 24.0
omega_max_demo = 1.0
alpha_demo = 0.5

U_demo = build_effective_unitary(
    T=T_demo,
    omega_max=omega_max_demo,
    alpha=alpha_demo,
    V=V,
    delta0=0.0,
)

plt.figure(figsize=(6, 5))
im = plt.imshow(np.abs(U_demo), origin='lower', aspect='equal')
plt.xticks(range(4), basis_labels)
plt.yticks(range(4), basis_labels)
plt.xlabel('Input basis state')
plt.ylabel('Output basis amplitude')
plt.title(r'$|U_{eff}|$ for a constrained adiabatic candidate')
plt.colorbar(im, label='magnitude')
plt.tight_layout()
plt.show()


## Gate-time scan at fixed constrained schedule

In [ ]:
T_vals = np.linspace(8.0, 42.0, 40)
omega_max = 1.0
alpha = 0.5

raw_vals = []
pc_vals = []
phi_vals = []
leak_vals = []
score_vals = []

for T in T_vals:
    U_eff = build_effective_unitary(
        T=T,
        omega_max=omega_max,
        alpha=alpha,
        V=V,
        delta0=0.0,
    )
    raw_vals.append(process_fidelity(U_eff))
    pc_vals.append(phase_corrected_fidelity(U_eff))
    phi_vals.append(entangling_phase(U_eff) / np.pi)
    leak_vals.append(leakage_metric(U_eff))
    score_vals.append(composite_score(U_eff))


In [ ]:
plt.figure(figsize=(7.5, 4.8))
plt.plot(T_vals, score_vals, label='composite score')
plt.plot(T_vals, pc_vals, label='phase-corrected fidelity')
plt.plot(T_vals, raw_vals, label='raw fidelity')
plt.xlabel('Total gate time T')
plt.ylabel('Score / fidelity')
plt.title('Constraint-aware adiabatic gate-time scan')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(T_vals, phi_vals)
plt.axhline(1.0, linestyle='--', label='ideal +1')
plt.axhline(-1.0, linestyle='--', color='gray', label='ideal -1')
plt.xlabel('Total gate time T')
plt.ylabel(r'Entangling phase / $\pi$')
plt.title('Entangling phase under constrained schedules')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(T_vals, leak_vals)
plt.xlabel('Total gate time T')
plt.ylabel('Leakage norm')
plt.title('Leakage under constrained schedules')
plt.tight_layout()
plt.show()


## 2D scan over $(T, \alpha)$

In [ ]:
alpha_vals = np.linspace(0.2, 1.2, 24)
T_grid = np.linspace(8.0, 42.0, 28)

raw_map = np.zeros((len(alpha_vals), len(T_grid)))
pc_map = np.zeros((len(alpha_vals), len(T_grid)))
phi_map = np.zeros((len(alpha_vals), len(T_grid)))
leak_map = np.zeros((len(alpha_vals), len(T_grid)))
score_map = np.zeros((len(alpha_vals), len(T_grid)))

omega_max = 1.0

for i, alpha in enumerate(alpha_vals):
    for j, T in enumerate(T_grid):
        U_eff = build_effective_unitary(
            T=T,
            omega_max=omega_max,
            alpha=alpha,
            V=V,
            delta0=0.0,
        )
        raw_map[i, j] = process_fidelity(U_eff)
        pc_map[i, j] = phase_corrected_fidelity(U_eff)
        phi_map[i, j] = entangling_phase(U_eff) / np.pi
        leak_map[i, j] = leakage_metric(U_eff)
        score_map[i, j] = composite_score(U_eff)


In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(score_map, origin='lower', aspect='auto', extent=[T_grid[0], T_grid[-1], alpha_vals[0], alpha_vals[-1]])
plt.contour(T_grid, alpha_vals, score_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title(r'Composite score over $(T, \alpha)$')
plt.colorbar(im, label='composite score')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(pc_map, origin='lower', aspect='auto', extent=[T_grid[0], T_grid[-1], alpha_vals[0], alpha_vals[-1]])
plt.contour(T_grid, alpha_vals, pc_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title(r'Phase-corrected fidelity over $(T, \alpha)$')
plt.colorbar(im, label='phase-corrected fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(phi_map, origin='lower', aspect='auto', extent=[T_grid[0], T_grid[-1], alpha_vals[0], alpha_vals[-1]], vmin=-1, vmax=1)
plt.contour(T_grid, alpha_vals, phi_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title(r'Entangling phase / $\pi$ over $(T, \alpha)$')
plt.colorbar(im, label=r'$\phi_{ent}/\pi$')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(leak_map, origin='lower', aspect='auto', extent=[T_grid[0], T_grid[-1], alpha_vals[0], alpha_vals[-1]])
plt.contour(T_grid, alpha_vals, leak_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $\alpha$')
plt.title(r'Leakage over $(T, \alpha)$')
plt.colorbar(im, label='leakage')
plt.tight_layout()
plt.show()


## Best constrained adiabatic candidate

In [ ]:
best_idx = np.unravel_index(np.argmax(score_map), score_map.shape)
best_alpha = alpha_vals[best_idx[0]]
best_T = T_grid[best_idx[1]]

U_best = build_effective_unitary(
    T=best_T,
    omega_max=omega_max,
    alpha=best_alpha,
    V=V,
    delta0=0.0,
)

print(f'Best constrained candidate: T = {best_T:.4f}, alpha = {best_alpha:.4f}')
print(f'  raw fidelity:             {process_fidelity(U_best):.6f}')
print(f'  phase-corrected fidelity: {phase_corrected_fidelity(U_best):.6f}')
print(f'  entangling phase / pi:    {entangling_phase(U_best)/np.pi:.6f}')
print(f'  leakage:                  {leakage_metric(U_best):.6f}')
print(f'  composite score:          {composite_score(U_best):.6f}')


## Interpretation

This notebook adds a simple but important idea:

> use a **constraint-aware schedule** instead of a generic one.

The detuning now scales with the interaction strength, and the objective no longer treats all high-fidelity points equally. Instead, it favors candidates that are simultaneously:

- low leakage,
- phase aligned,
- close to a diagonal controlled-phase operation.

This should produce cleaner ridges than Notebook 11 and make Notebook 12 optimization much more stable.


## Optional next steps

Natural next upgrades are:

- optimize both $\Omega_{max}$ and $\alpha$ together,
- add a nonzero $\Delta_0$ offset,
- refine the top ridge with a denser local scan,
- move directly into gradient-based optimization in Notebook 12.
